# 🐾 Animal Sound Generator — Colab Training v2

**May 2026 fix:** skip_dropout=1.0, embed_dim=256, stronger FiLM, no Griffin-Lim by default.

| GPU | VRAM | AE Time | VAE Time |
|-----|------|---------|----------|
| T4 (free) | 16 GB | ~2 hrs | ~2 hrs |
| L4 (pro) | 24 GB | ~1.2 hrs | ~1.2 hrs |

### Before running:
1. Upload `animal_audio.tar.gz` to Google Drive root (`MyDrive/`)
2. Runtime → Change runtime type → **L4 GPU** (or T4)
3. Run cells **top to bottom**

In [ ]:
# @title 1. Setup

from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/grindydev/animal_sound_generator.git /content/animal_sound_generator
%cd /content/animal_sound_generator
!git pull  # get latest fixes

!pip install -q torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy matplotlib pandas scikit-learn librosa soundfile tqdm

!mkdir -p models/autoencoder_checkpoints/train
!mkdir -p models/vae_checkpoints/train

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
# @title 2. Load Dataset from Drive

import os, tarfile

LOCAL = '/content/animal_sound_generator/data'
TAR = '/content/drive/MyDrive/animal_audio.tar.gz'

if os.path.isdir(os.path.join(LOCAL, 'animal_audio')):
    print('✅ Data already loaded')
elif os.path.exists(TAR):
    print('📂 Extracting (~30s)...')
    os.makedirs(LOCAL, exist_ok=True)
    with tarfile.open(TAR, 'r:gz') as tf:
        tf.extractall(path=LOCAL, filter='data')
    nested = os.path.join(LOCAL, 'data', 'animal_audio')
    if os.path.isdir(nested):
        !mv {nested} {LOCAL}/animal_audio && rmdir {LOCAL}/data
    print('✅ Done!')
else:
    print('❌ animal_audio.tar.gz not found in Drive root!')

!ls data/animal_audio/

In [ ]:
# @title 3. Copy Models from Drive (resume training)

DRIVE = '/content/drive/MyDrive/animal_sound_generator/models'

if os.path.isdir(DRIVE):
    !cp -r {DRIVE}/* models/ 2>/dev/null
    !ls -lh models/*.pth 2>/dev/null || echo 'No .pth files'
    print('✅ Checkpoints restored')
else:
    print('ℹ️  No Drive checkpoints — training from scratch')

In [ ]:
# @title 4a. Train Autoencoder (~1-2 hrs)

# Config: lr=1e-3, batch=16, base_ch=32 → 149M params
# SKIP if best_autoencoder_train.pth exists from Drive
!python src/vae/train_ae.py

In [ ]:
# @title 4b. DELETE old VAE checkpoint (if retraining with new fix)

# The old VAE was trained with skip_dropout=0.5 and produces noise.
# Remove it so finetune starts fresh.
!rm -f models/best_vae_finetune_train.pth
!rm -rf models/vae_checkpoints/
!mkdir -p models/vae_checkpoints/train
print('✅ Old VAE checkpoint deleted')
print('   New config: skip_dropout=1.0, embed_dim=256, class_loss=1.0')

In [ ]:
# @title 4c. Train VAE with Fix (~1-2 hrs) ⭐ DO THIS

# Fix plan v2: decoder learns WITHOUT skip connections
#   skip_dropout: 0.5 → 1.0  (never see skips during training)
#   embed_dim:    128 → 256  (stronger class signal via FiLM)
#   class_loss:   0.5 → 1.0  (classifier pushes harder)
!python src/vae/finetune.py

In [ ]:
# @title 5. (Optional) Train Diffusion (~3 hrs)
!python src/diffusion/train.py

---
## 💾 Save All Models to Drive

In [ ]:
# @title 💾 Save All to Drive

DRIVE = '/content/drive/MyDrive/animal_sound_generator/models'
!mkdir -p {DRIVE}

import os
for f in ['best_autoencoder_train.pth', 'best_vae_finetune_train.pth',
          'best_audio_cnn_train.pth', 'diffusion_unet_train_best.pth',
          'hifigan_generator_train_best.pth']:
    path = f'models/{f}'
    if os.path.exists(path):
        !cp {path} {DRIVE}/
        print(f'  ✅ {f} ({os.path.getsize(path)/1e6:.0f} MB)')

for d in ['autoencoder_checkpoints', 'vae_checkpoints', 'diffusion_checkpoints']:
    if os.path.isdir(f'models/{d}'):
        !cp -r models/{d} {DRIVE}/
        print(f'  ✅ {d}/')

print(f'\n📂 All saved to {DRIVE}/')
!ls -lh {DRIVE}/

---
## 🔁 Resume After Timeout

If Colab disconnects, run this cell THEN re-run the training cell you were on.

In [ ]:
# @title 🔁 Resume

DRIVE_M = '/content/drive/MyDrive/animal_sound_generator/models'
if os.path.isdir(DRIVE_M):
    !cp -r {DRIVE_M}/* models/ 2>/dev/null
    print('✅ Checkpoints restored')

# Re-extract data
import tarfile, os
TAR = '/content/drive/MyDrive/animal_audio.tar.gz'
LOCAL = '/content/animal_sound_generator/data'
if not os.path.isdir(os.path.join(LOCAL, 'animal_audio')) and os.path.exists(TAR):
    os.makedirs(LOCAL, exist_ok=True)
    with tarfile.open(TAR, 'r:gz') as tf:
        tf.extractall(path=LOCAL, filter='data')
    nested = os.path.join(LOCAL, 'data', 'animal_audio')
    if os.path.isdir(nested):
        !mv {nested} {LOCAL}/animal_audio && rmdir {LOCAL}/data
    print('✅ Data re-extracted')
print('Ready — re-run your training cell')


---
## 🎧 Generate & Listen

In [ ]:
# @title 🎧 Generate

# No Griffin-Lim by default. Temperature 0.5.
!python src/generate.py --label Dog --no-diff --temperature 0.5

from IPython.display import Audio, display
import glob
wavs = sorted(glob.glob('generated_audio/*.wav'))
if wavs:
    for w in wavs[-3:]:  # last 3
        display(Audio(w, rate=22050))
        print(f'🔊 {w}')
else:
    print('No audio generated')